# Test vs Trained Results

This notebook is a simple checkpoint to:
- show baseline vs trained result assets,
- show dashboard screenshot,
- show all images from assets/results,
- show available logs and evaluation files,
- load the trained adapter and run a basic generation test.

> ## Theory Box
> A trained policy should outperform baseline on the same split because:
> 1. SFT improves action-format reliability.
> 2. GRPO optimizes policy behavior against environment reward signals.
>
> Original curve references:
> - Training reward curve: [assets/results/training_reward_curve.png](./assets/results/training_reward_curve.png)
> - Baseline vs trained reward: [assets/results/baseline_vs_trained_reward.png](./assets/results/baseline_vs_trained_reward.png)
> - Training Space: https://huggingface.co/spaces/samarthdave0305/my-training

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path('.').resolve()
RESULTS_DIR = ROOT / 'assets' / 'results'

EXPECTED_IMAGES = {
    'training_loss': RESULTS_DIR / 'training_loss_curve.png',
    'training_reward': RESULTS_DIR / 'training_reward_curve.png',
    'baseline_vs_trained_reward': RESULTS_DIR / 'baseline_vs_trained_reward.png',
    'demo_dashboard': RESULTS_DIR / 'demo_dashboard.png',
}

CAPTIONS = {
    'training_loss': 'SFT/GRPO loss over training step.',
    'training_reward': 'GRPO reward over training step.',
    'baseline_vs_trained_reward': 'Baseline policy versus trained policy on the same evaluation split.',
    'demo_dashboard': 'Live CascadeGuard Space dashboard.',
}

LINKS = {
    'live_space': 'https://huggingface.co/spaces/samarthdave0305/cascade-failure-env',
    'training_space': 'https://huggingface.co/spaces/samarthdave0305/my-training',
    'trained_adapter': 'https://huggingface.co/samarthdave0305/cascadeguard-trained',
}

print('ROOT:', ROOT)
print('RESULTS_DIR:', RESULTS_DIR)
print('Links:')
for k, v in LINKS.items():
    print(f'  - {k}: {v}')

In [ ]:
print('Expected result files:')
for key, path in EXPECTED_IMAGES.items():
    status = 'FOUND' if path.exists() else 'MISSING'
    print(f'- {key:28s} {status:8s} {path.as_posix()}')

In [ ]:
def show_image_with_caption(path: Path, caption: str):
    if path.exists():
        display(Image(filename=str(path)))
        display(Markdown(f'Caption: {caption}'))
    else:
        display(Markdown(f'**Missing image:** {path.as_posix()}'))

display(Markdown('## Requested Result Images'))
show_image_with_caption(EXPECTED_IMAGES['baseline_vs_trained_reward'], CAPTIONS['baseline_vs_trained_reward'])
show_image_with_caption(EXPECTED_IMAGES['demo_dashboard'], CAPTIONS['demo_dashboard'])

display(Markdown('## Additional Curves'))
show_image_with_caption(EXPECTED_IMAGES['training_loss'], CAPTIONS['training_loss'])
show_image_with_caption(EXPECTED_IMAGES['training_reward'], CAPTIONS['training_reward'])

In [ ]:
display(Markdown('## All Images Under assets/results'))
if RESULTS_DIR.exists():
    exts = {'.png', '.jpg', '.jpeg', '.webp'}
    all_images = sorted([p for p in RESULTS_DIR.rglob('*') if p.is_file() and p.suffix.lower() in exts])
    print(f'Found {len(all_images)} image(s).')
    for img in all_images:
        display(Markdown(f'### {img.relative_to(ROOT).as_posix()}'))
        display(Image(filename=str(img)))
else:
    print('assets/results directory not found.')

In [ ]:
display(Markdown('## Logs and Evaluation Artifacts'))

candidate_text_logs = [
    ROOT / 'cascadeguard_training_live.log',
    Path('/home/user/app/cascadeguard_training_live.log'),
]

candidate_json_logs = [
    ROOT / 'training_curves' / 'training_log.json',
    Path('/home/user/app/cascade_gaurd_openEnv/training_curves/training_log.json'),
]

candidate_csvs = [
    ROOT / 'cascadeguard_eval_results.csv',
    Path('/home/user/app/cascade_gaurd_openEnv/cascadeguard_eval_results.csv'),
]

for p in candidate_text_logs:
    if p.exists():
        print(f'Found text log: {p}')
        lines = p.read_text(encoding='utf-8', errors='ignore').splitlines()
        print('--- tail(60) ---')
        for ln in lines[-60:]:
            print(ln)

for p in candidate_json_logs:
    if p.exists():
        print(f'Found JSON log: {p}')
        data = json.loads(p.read_text(encoding='utf-8', errors='ignore'))
        if isinstance(data, list) and data:
            df = pd.DataFrame(data)
            display(df.tail(20))
        else:
            print('JSON log exists but is empty or not a list.')

for p in candidate_csvs:
    if p.exists():
        print(f'Found CSV: {p}')
        df = pd.read_csv(p)
        display(df.tail(20))

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = os.getenv('BASE_MODEL', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B')
ADAPTER_REPO = os.getenv('ADAPTER_REPO', 'samarthdave0305/cascadeguard-trained')
LOCAL_ADAPTER = ROOT / 'training_outputs' / 'grpo' / 'final'

# Kaggle dual-GPU friendly defaults
USE_4BIT = os.getenv('USE_4BIT', '1') == '1'
RESERVE_GIB_PER_GPU = int(os.getenv('RESERVE_GIB_PER_GPU', '2'))

adapter_source = str(LOCAL_ADAPTER) if LOCAL_ADAPTER.exists() else ADAPTER_REPO
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

gpu_count = torch.cuda.device_count()
print('Base model    :', BASE_MODEL)
print('Adapter source:', adapter_source)
print('CUDA available:', torch.cuda.is_available())
print('GPU count     :', gpu_count)
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | VRAM {p.total_memory / (1024**3):.1f} GiB')

tokenizer = None
model = None
max_memory = None
device_map = 'auto' if gpu_count > 0 else None

if gpu_count > 0:
    max_memory = {}
    for i in range(gpu_count):
        total_gib = int(torch.cuda.get_device_properties(i).total_memory // (1024**3))
        alloc_gib = max(4, total_gib - RESERVE_GIB_PER_GPU)
        max_memory[i] = f'{alloc_gib}GiB'
    max_memory['cpu'] = '64GiB'
    print('max_memory map:', max_memory)

try:
    tokenizer = AutoTokenizer.from_pretrained(adapter_source, trust_remote_code=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = dict(
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    torch_dtype=dtype,
    device_map=device_map,
    max_memory=max_memory,
    attn_implementation='eager',
)

if USE_4BIT:
    try:
        from transformers import BitsAndBytesConfig
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
        )
        print('4-bit quantization enabled.')
    except Exception as e:
        print('4-bit quantization unavailable, falling back to normal load:', e)

try:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
    model = PeftModel.from_pretrained(model, adapter_source)
    model.eval()
    print('Model loaded successfully.')
except Exception as e:
    print('Model load failed:', e)
    print('Tip: on Kaggle, keep USE_4BIT=1 and run with both GPUs enabled.')

In [ ]:
if model is None or tokenizer is None:
    print('Skipping generation because model/tokenizer is not loaded.')
else:
    prompt = (
        'You are a CascadeGuard agent. Current failures: POWER_DIST_1, WATER_PUMP_2. '
        'Critical node: HOSP_1. Budget: 6. Return one best next action in JSON.'
    )
    inputs = tokenizer(prompt, return_tensors='pt')

    # For sharded multi-GPU models, place inputs on the first CUDA shard.
    target_device = None
    if hasattr(model, 'hf_device_map') and isinstance(model.hf_device_map, dict):
        for v in model.hf_device_map.values():
            if isinstance(v, str) and v.startswith('cuda'):
                target_device = v
                break
    if target_device is None and torch.cuda.is_available():
        target_device = 'cuda:0'

    if target_device is not None:
        inputs = {k: v.to(target_device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    print(decoded)